Dataset
------------
First, import the recordlinkage module and load the Krebs register data. The dataset contains 5749132 compared record pairs and has the following variables: first name, last name, sex, birthday, birth month, birth year and zip code. The Krebs register contains len(krebs_true_links) == 20931 matching record pairs.

In [1]:
!pip install recordlinkage

  Using cached recordlinkage-0.16-py3-none-any.whl.metadata (8.1 kB)
Using cached recordlinkage-0.16-py3-none-any.whl (926 kB)



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas
import recordlinkage as rl
from recordlinkage.datasets import load_krebsregister

Most classifiers can not handle comparison vectors with missing values. To prevent issues with the classification algorithms, we convert the missing values into disagreeing comparisons (using argument missing_values=0). For example if the comparison between two names is missing, the similiraty will be set to 0.
This approach for handling missing values in comparison features is widely used in record linkage applications.

In [3]:
krebs_X, krebs_true_links = load_krebsregister(missing_values=0)
krebs_X

Data download succesfull.


,,cmp_firstname1,cmp_firstname2,cmp_lastname1,cmp_lastname2,cmp_sex,cmp_birthday,cmp_birthmonth,cmp_birthyear,cmp_zipcode
id1,id2,,,,,,,,,
22161,38467,1.000000,0.0,0.142857,0.0,1,0.0,1.0,0.0,0.0
38713,75352,0.000000,0.0,0.571429,0.0,1,0.0,0.0,0.0,0.0
13699,32825,0.166667,0.0,0.000000,0.0,0,1.0,1.0,1.0,0.0
22709,37682,0.285714,0.0,1.000000,0.0,1,0.0,0.0,0.0,0.0
2342,69060,0.250000,0.0,0.125000,0.0,1,1.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...
52124,53629,1.000000,0.0,0.285714,0.0,1,0.0,0.0,1.0,0.0
30007,76846,0.750000,0.0,0.000000,0.0,1,1.0,0.0,0.0,0.0
50546,59461,0.750000,0.0,0.000000,0.0,1,0.0,1.0,0.0,0.0


In [4]:
krebs_X.describe()

,cmp_firstname1,cmp_firstname2,cmp_lastname1,cmp_lastname2,cmp_sex,cmp_birthday,cmp_birthmonth,cmp_birthyear,cmp_zipcode
count,5.749132e+06,5.749132e+06,5.749132e+06,5.749132e+06,5.749132e+06,5.749132e+06,5.749132e+06,5.749132e+06,5.749132e+06
mean,7.127776e-01,1.623376e-02,3.156278e-01,1.364674e-04,9.550014e-01,2.244342e-01,4.887877e-01,2.227178e-01,5.516311e-03
std,3.888388e-01,1.251994e-01,3.342336e-01,1.008120e-02,2.073011e-01,4.172092e-01,4.998743e-01,4.160704e-01,7.406674e-02
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.857143e-01,0.000000e+00,1.000000e-01,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,1.000000e+00,0.000000e+00,1.818182e-01,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.000000e+00,0.000000e+00,4.285714e-01,0.000000e+00,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
max,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


In [5]:
krebs_true_links

MultiIndex([(89874, 89876),
            (79126, 84983),
            (40350, 83715),
            (75394, 92002),
            (23323, 27823),
            (31059, 72216),
            (28464, 69899),
            (33613, 64971),
            (23546, 27978),
            (29922, 46075),
            ...
            (62697, 62705),
            (82751, 82753),
            (45892, 59324),
            (33619, 40411),
            (28562, 43893),
            (26162, 27408),
            (71973, 71976),
            (23925, 29112),
            (57711, 99655),
            (18795, 41061)],
           names=['id1', 'id2'], length=20931)

As described before, supervised learning algorithms do need training data. Training data is data for which the true match status is known for each comparison vector. In the example in this section, we consider that the true match status of the first 50,000 record pairs of the Krebs register data is known.

In [6]:
golden_pairs = krebs_X[0:50000]
# 183 matching pairs
golden_matches_index = golden_pairs.index.intersection(krebs_true_links)
golden_matches_index

MultiIndex([(89874, 89876),
            (79126, 84983),
            (40350, 83715),
            (75394, 92002),
            (23323, 27823),
            (31059, 72216),
            (28464, 69899),
            (33613, 64971),
            (23546, 27978),
            (29922, 46075),
            ...
            (19225, 27571),
            (20087, 23175),
            (10844, 10846),
            (23474, 41070),
            (28201, 44177),
            (48837, 63106),
            (  172, 59437),
            (28110, 28559),
            ( 5896, 39217),
            (34908, 38121)],
           names=['id1', 'id2'], length=183)

Logistic regression
--------
The recordlinkage.LogisticRegressionClassifier classifier is an application of the logistic regression model. This supervised learning method is one of the oldest classification algorithms used in record linkage. In situations with enough training data, the algorithm gives relatively good results.

In [7]:
# Initialize the classifier
logreg = rl.LogisticRegressionClassifier()

# Train the classifier
logreg.fit(golden_pairs, golden_matches_index)
#print ("Intercept: ", logreg.intercept)
#print ("Coefficients: ", logreg.coefficients)

Predict the match status for all record pairs.

In [8]:
result_logreg = logreg.predict(krebs_X)
len(result_logreg)

20029

Compute the confusion matrix.
-----------

The confusion matrix is of the following form:
                      
                          True Positives          True Negatives
    Predicted Positives   True Positives (TP)     False Positives (FP)
    Predicted Negatives   False Negatives (FN)    True Negatives (TN)


In [9]:
rl.confusion_matrix(krebs_true_links, result_logreg, len(krebs_X))

array([[  20023,     908],
       [      6, 5728195]])

Evaluation Metrics
---------
Evaluation of classifications plays an important role in record linkage. 
Below, the quality of the classification model is expressed in terms of accuracy, recall and F-score based on true positives, false positives, true negatives and false negatives.

In [10]:
#The F-score for this prediction
rl.fscore(krebs_true_links, result_logreg)

0.977685546875